# Regresión Lineal Simple

## Caso de uso: Predicción del Coste de un Incidente de Seguridad

---

### ¿Qué es la Regresión Lineal?

La **Regresión Lineal** es un algoritmo de aprendizaje supervisado que busca modelar la relación entre:
- Una **variable independiente** (entrada, `x`): la causa o el predictor.
- Una **variable dependiente** (salida, `y`): el efecto que queremos predecir.

La idea es ajustar la **recta que mejor describe** esa relación usando los datos históricos disponibles. La ecuación del modelo es:

$$H(x) = \theta_0 + \theta_1 \cdot x$$

Donde:
- $\theta_0$ (**sesgo** o *bias*): el valor de `y` cuando `x = 0`. Representa el coste base independiente del número de equipos.
- $\theta_1$ (**pendiente**): cuánto aumenta `y` por cada unidad adicional de `x`. Representa el coste adicional por cada equipo afectado.

---

### Problema a resolver

Queremos predecir el **coste económico de un incidente de seguridad** en función del **número de equipos afectados**. Trabajaremos con un dataset generado de forma aleatoria (con una semilla fija para reproducibilidad).

El flujo del ejercicio será:

1. **Importar librerías**
2. **Generar el dataset sintético**
3. **Visualizar** los datos para entender su distribución
4. **Escalar** los datos a unidades realistas (equipos y euros)
5. **Entrenar** el modelo de Regresión Lineal
6. **Predecir** el coste para un nuevo incidente

---
## Paso 0 — Importar librerías

Cargamos las herramientas que vamos a necesitar:

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Generar números aleatorios y operar con arrays |
| `pandas` | Organizar los datos en una tabla (DataFrame) |
| `matplotlib.pyplot` | Crear gráficas |
| `sklearn.linear_model.LinearRegression` | Entrenar el modelo de regresión |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

%matplotlib inline

---
## Paso 1 — Generar el Dataset

Como no disponemos de datos reales, generamos un dataset sintético que **simula** la relación entre equipos afectados y coste del incidente.

La fórmula con la que generamos los datos es:

$$y = 4 + 3x + \epsilon$$

- $x$: número de equipos (normalizado, entre 0 y 2)
- $4$: coste base de cualquier incidente ($\theta_0$ real)
- $3$: incremento de coste por equipo ($\theta_1$ real)
- $\epsilon \sim \mathcal{N}(0,1)$: **ruido gaussiano** que simula la variabilidad natural

> **¿Por qué añadir ruido?** En el mundo real, dos incidentes con el mismo número de equipos nunca tienen exactamente el mismo coste. Factores como el tipo de ataque, la criticidad de los sistemas o el tiempo de respuesta también influyen. El ruido modela esa incertidumbre.

> **`np.random.seed(42)`** fija la semilla aleatoria para que los resultados sean **reproducibles**: siempre obtendrás el mismo dataset al ejecutar el código.

In [ ]:
np.random.seed(42)  # Fija la semilla para reproducibilidad

x = 2 * np.random.rand(100, 1)          # 100 valores aleatorios en [0, 2]
y = 4 + 3 * x + np.random.randn(100, 1) # Relación lineal + ruido gaussiano

print(f"Número de ejemplos (m): {len(x)}")
print(f"Rango de x: [{x.min():.3f}, {x.max():.3f}]")
print(f"Rango de y: [{y.min():.3f}, {y.max():.3f}]")
print(f"\nParámetros REALES usados para generar los datos:")
print(f"  θ₀ (coste base) = 4")
print(f"  θ₁ (pendiente)  = 3")
print(f"\nEl modelo intentará recuperar estos parámetros a partir de los datos.")

---
## Paso 2 — Visualizar los Datos (escala normalizada)

Antes de entrenar el modelo, es fundamental **explorar visualmente** los datos.

En esta primera gráfica los datos están en la escala original (normalizada). Observa:
- La **tendencia lineal positiva**: a más equipos afectados, mayor coste.
- La **dispersión alrededor de la recta**: efecto del ruido gaussiano añadido.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x, y, color='steelblue', alpha=0.7, label='Datos generados')
plt.xlabel("Equipos afectados (u/1.000)")
plt.ylabel("Coste del incidente (u/10.000 €)")
plt.title("Relación entre equipos afectados y coste del incidente\n(escala normalizada)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

**Conclusión visual:** Se observa una tendencia lineal positiva clara. La dispersión refleja el ruido gaussiano incorporado al dataset, imitando la variabilidad natural de los datos reales.

---
## Paso 3 — Escalar los Datos a Unidades Reales

Los valores generados son abstractos (entre 0 y ~2). Para que el problema sea más **interpretable**, los convertimos a unidades realistas:

- **Equipos afectados** → multiplicamos por 1.000 → rango: ~0 a ~2.000 equipos
- **Coste del incidente** → multiplicamos por 10.000 → rango: ~30.000 a ~110.000 €

Usamos `.astype(int)` para trabajar con valores enteros (más naturales para contar equipos y euros).

In [ ]:
# Crear DataFrame con los datos originales
df = pd.DataFrame({
    'n_equipos_afectados': x.flatten(),
    'coste_incidente': y.flatten()
})

print("Datos antes de escalar (primeras 5 filas):")
print(df.head())

In [ ]:
# Escalar a unidades reales
df['n_equipos_afectados'] = (df['n_equipos_afectados'] * 1000).astype(int)
df['coste_incidente']     = (df['coste_incidente'] * 10000).astype(int)

print("Datos después de escalar (primeras 5 filas):")
print(df.head())
print(f"\nEstadísticas básicas:")
print(df.describe().round(0))

### Visualización con unidades reales

La misma gráfica anterior, ahora con los ejes en unidades comprensibles.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['n_equipos_afectados'], df['coste_incidente'],
            color='steelblue', alpha=0.7, label='Datos históricos')
plt.xlabel("Equipos afectados")
plt.ylabel("Coste del incidente (€)")
plt.title("Relación entre equipos afectados y coste del incidente\n(unidades reales)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

**Conclusión:** La tendencia es idéntica a la gráfica anterior, ahora en unidades reales: entre **0 y ~2.000 equipos** y costes de **~30.000 a ~110.000 €**. La variabilidad entre incidentes con el mismo número de equipos refleja la influencia de factores no capturados en el modelo.

---
## Paso 4 — Construir y Entrenar el Modelo

### ¿Qué hace `LinearRegression` de scikit-learn?

El modelo busca los parámetros $\theta_0$ y $\theta_1$ que **minimizan el Error Cuadrático Medio (MSE)** entre las predicciones y los valores reales:

$$\text{MSE} = \frac{1}{m} \sum_{i=1}^{m} \left( H(x^{(i)}) - y^{(i)} \right)^2$$

Scikit-learn lo resuelve usando la **ecuación normal** (solución analítica exacta), sin necesidad de iterar como haría el gradiente descendente.

### ¿Qué esperamos obtener?

Como generamos los datos con $\theta_0 = 4$ y $\theta_1 = 3$ (en escala normalizada), el modelo debería recuperar valores cercanos a:
- $\theta_0 \approx 4 \times 10.000 = 40.000$ €  (escalado)
- $\theta_1 \approx 3 \times \frac{10.000}{1.000} = 30$ € por equipo  (escalado)

In [ ]:
# Definir variables X (entrada) e y (salida)
X_train = df[['n_equipos_afectados']]  # Variable independiente (matriz 2D requerida por sklearn)
y_train = df['coste_incidente']        # Variable dependiente

# Crear y entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Extraer los parámetros aprendidos
theta_0 = round(modelo.intercept_)
theta_1 = round(modelo.coef_[0], 2)

print("═" * 45)
print("  PARÁMETROS APRENDIDOS POR EL MODELO")
print("═" * 45)
print(f"  θ₀ (coste base, bias):         {theta_0:>10,} €")
print(f"  θ₁ (coste por equipo):  {theta_1:>14.2f} €/equipo")
print("═" * 45)
print(f"\n  Ecuación del modelo:")
print(f"  Coste = {theta_0:,} + {theta_1} × (n_equipos)")
print()
print("  Interpretación:")
print(f"  • Cualquier incidente tiene un coste mínimo de {theta_0:,} €")
print(f"    independientemente del número de equipos.")
print(f"  • Cada equipo adicional afectado incrementa el")
print(f"    coste en aproximadamente {theta_1} €.")

### Visualización de la recta de regresión

Dibujamos la recta ajustada sobre los datos históricos. Solo necesitamos dos puntos para trazar una línea: el mínimo y el máximo de `x`.

In [ ]:
# Calcular dos puntos para trazar la recta (mín y máx de x)
X_linea = pd.DataFrame(
    [[df['n_equipos_afectados'].min()],
     [df['n_equipos_afectados'].max()]],
    columns=['n_equipos_afectados']
)
y_linea = modelo.predict(X_linea)

# Graficar
plt.figure(figsize=(8, 5))
plt.scatter(df['n_equipos_afectados'], df['coste_incidente'],
            color='steelblue', alpha=0.7, label='Datos históricos')
plt.plot(X_linea, y_linea,
         color='green', linewidth=2.5, label=f'Modelo: Coste = {theta_0:,} + {theta_1}·x')
plt.xlabel("Equipos afectados")
plt.ylabel("Coste del incidente (€)")
plt.title("Regresión Lineal: Coste del incidente vs. Equipos afectados")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Interpretación de la gráfica:**

La **línea verde** representa la función hipótesis $H(x) = \theta_0 + \theta_1 x$ ajustada por el modelo. Es la recta que **minimiza la suma de los errores al cuadrado** entre cada punto azul y su valor predicho.

Los puntos se distribuyen a ambos lados de la recta sin un patrón sistemático, lo que indica un **buen ajuste** del modelo lineal a los datos.

---
## Paso 5 — Realizar una Predicción

### Escenario

> Se ha detectado un incidente de seguridad que ha comprometido **1.500 equipos**. ¿Cuál es el coste estimado?

El modelo calculará:

$$\text{Coste estimado} = \theta_0 + \theta_1 \times 1.500$$

In [ ]:
# Nuevo ejemplo a predecir: 1.500 equipos afectados
n_equipos_nuevo = 1500
X_nuevo = pd.DataFrame([[n_equipos_nuevo]], columns=['n_equipos_afectados'])

# Obtener la predicción
coste_predicho = modelo.predict(X_nuevo)[0]

print("═" * 40)
print("  PREDICCIÓN DEL MODELO")
print("═" * 40)
print(f"  Equipos afectados:  {n_equipos_nuevo:>8,}")
print(f"  Coste estimado:     {int(coste_predicho):>8,} €")
print("═" * 40)
print(f"\n  Cálculo manual:")
print(f"  {theta_0:,} + {theta_1} × {n_equipos_nuevo:,} = {theta_0 + theta_1 * n_equipos_nuevo:,.0f} €")

### Visualización de la predicción

In [ ]:
plt.figure(figsize=(9, 5))

# Datos históricos
plt.scatter(df['n_equipos_afectados'], df['coste_incidente'],
            color='steelblue', alpha=0.7, label='Datos históricos')

# Recta de regresión
plt.plot(X_linea, y_linea,
         color='green', linewidth=2.5, label='Función hipótesis')

# Nueva predicción
plt.scatter(X_nuevo, coste_predicho,
            color='red', marker='X', s=200, zorder=5,
            label=f'Predicción: {int(coste_predicho):,} €')

# Líneas guía hacia la predicción
plt.axvline(x=n_equipos_nuevo, color='red', linestyle=':', alpha=0.4)
plt.axhline(y=coste_predicho, color='red', linestyle=':', alpha=0.4)

plt.xlabel("Equipos afectados")
plt.ylabel("Coste del incidente (€)")
plt.title(f"Predicción del coste para {n_equipos_nuevo:,} equipos afectados")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Interpretación:**

La **X roja** muestra la predicción del modelo sobre la recta de regresión. Sin embargo, observa que los **puntos azules cercanos a x = 1.500** tienen costes reales que van desde ~70.000 € hasta ~102.000 €.

Esto ilustra una **limitación importante** del modelo: al usar una única variable (número de equipos), no captura toda la variabilidad real. Factores como el tipo de ataque, la criticidad de los sistemas comprometidos, el tiempo de detección o la respuesta del equipo de seguridad también influyen en el coste final del incidente.

---
## Resumen

| Paso | Descripción | Resultado |
|------|-------------|----------|
| 1 | Generación del dataset sintético | 100 ejemplos con relación lineal + ruido |
| 2 | Exploración visual | Tendencia lineal positiva confirmada |
| 3 | Escalado de datos | Unidades reales: equipos (0-2.000) y euros (30k-110k) |
| 4 | Entrenamiento del modelo | θ₀ ≈ 42.163 €, θ₁ ≈ 27,70 €/equipo |
| 5 | Predicción para 1.500 equipos | Coste estimado ≈ 83.716 € |

### Próximos pasos

Para mejorar este modelo se podría:
- Añadir más variables (tipo de ataque, criticidad, tiempo de respuesta) → **Regresión Lineal Múltiple**
- Evaluar el modelo con métricas como MSE, RMSE o R² en un conjunto de test
- Probar modelos más complejos si la relación no es perfectamente lineal